# Vision Transformer Patch Visualization
How does the **patch size** change the number of tokens, the spatial information and the feature representation of a ViT?

This notebook walks through the same steps as `src/vit_patch_visualization.py`, one at a time.

## 1. Imports and settings

In [1]:
import sys
sys.path.append('../src')
import numpy as np
import matplotlib.pyplot as plt
from vit_patch_visualization import (load_image, split_into_patches, patch_embedding,
    add_class_token_and_positions, mean_color_reconstruction, mse_psnr, pca_to_rgb,
    cosine_similarity_map)

PATCH_SIZES = [8, 16, 32, 56]
image = load_image('../dataset/sample_image.png')
plt.imshow(image); plt.title('Input image (224x224)'); plt.axis('off'); plt.show()

## 2. Step 1 - Split the image into patches
Tokens = (224 / patch)^2. Each patch is flattened into a vector of `patch x patch x 3` values.

In [2]:
for p in PATCH_SIZES:
    patches, grid = split_into_patches(image, p)
    print(f'patch {p:>2}: grid {grid}, tokens = {patches.shape[0]:>3}, values per patch = {patches.shape[1]}')

patch  8: grid (28, 28), tokens = 784, values per patch = 192
patch 16: grid (14, 14), tokens = 196, values per patch = 768
patch 32: grid (7, 7), tokens =  49, values per patch = 3072
patch 56: grid (4, 4), tokens =  16, values per patch = 9408


## 3. Step 2 - Patch embedding (linear projection)
Every flattened patch is multiplied by one weight matrix to get a 64-dimensional token. The weights are random because we only compare patch sizes; no training is done.

In [3]:
patches, grid = split_into_patches(image, 16)
emb = patch_embedding(patches)
seq = add_class_token_and_positions(emb)
print('patches :', patches.shape)
print('embedded:', emb.shape)
print('with CLS:', seq.shape)

patches : (196, 768)
embedded: (196, 64)
with CLS: (197, 64)


## 4. Patch grids

In [4]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4.4))
for ax, p in zip(axes, PATCH_SIZES):
    ax.imshow(image)
    for t in range(0, 225, p):
        ax.axhline(t-0.5, color='white', lw=0.6); ax.axvline(t-0.5, color='white', lw=0.6)
    ax.set_title(f'Patch {p}: {224//p}x{224//p} = {(224//p)**2} tokens'); ax.axis('off')
plt.show()

## 5. Spatial information kept
Replace every patch by its mean colour. The bigger the patch, the more detail is lost (lower PSNR).

In [5]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4.4))
for ax, p in zip(axes, PATCH_SIZES):
    rec = mean_color_reconstruction(image, p)
    mse, psnr = mse_psnr(image, rec)
    ax.imshow(rec); ax.set_title(f'Patch {p}  PSNR = {psnr:.1f} dB'); ax.axis('off')
plt.show()

## 6. Feature representation (PCA of the embeddings shown as colours)

In [6]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4.2))
for ax, p in zip(axes, PATCH_SIZES):
    pt, (gh, gw) = split_into_patches(image, p)
    rgb = pca_to_rgb(patch_embedding(pt)).reshape(gh, gw, 3)
    ax.imshow(rgb, interpolation='nearest'); ax.set_title(f'Patch {p}: {gh}x{gw} tokens'); ax.axis('off')
plt.show()

## 7. Which patches look alike? (cosine similarity to one query patch)

In [7]:
qy, qx = 70, 110
fig, axes = plt.subplots(1, 4, figsize=(16, 4.2))
for ax, p in zip(axes, PATCH_SIZES):
    pt, (gh, gw) = split_into_patches(image, p)
    e = patch_embedding(pt)
    sim = cosine_similarity_map(e, (qy//p)*gw + (qx//p)).reshape(gh, gw)
    ax.imshow(image); ax.imshow(np.kron(sim, np.ones((p, p))), cmap='jet', alpha=0.55, vmin=-1, vmax=1)
    ax.set_title(f'Patch {p}'); ax.axis('off')
plt.show()

## 8. Summary table and attention cost

In [8]:
print('Patch | Tokens | Seq len | Attention entries | PSNR (dB)')
for p in PATCH_SIZES:
    pt, g = split_into_patches(image, p)
    n = pt.shape[0] + 1
    _, psnr = mse_psnr(image, mean_color_reconstruction(image, p))
    print(f'{p:>5} | {pt.shape[0]:>6} | {n:>7} | {n*n:>17} | {psnr:.2f}')

Patch | Tokens | Seq len | Attention entries | PSNR (dB)
    8 |    784 |     785 |            616225 | 17.30
   16 |    196 |     197 |             38809 | 14.78
   32 |     49 |      50 |              2500 | 12.94
   56 |     16 |      17 |               289 | 12.03


## 9. Conclusion
* Smaller patch -> more tokens, finer detail, but attention cost grows with (tokens)^2.
* Larger patch -> fewer tokens and cheaper attention, but spatial detail is lost and each token mixes many different pixels.
* Patch 16 (the standard ViT-B/16 choice) is a middle ground.